# Three-SMU plotting dashboard

One read-only dashboard for saved accepted runs and a scan currently started by `python -m attodry_control.three_smu_cli run`. Live mode receives already-recorded memory samples from that CLI; this Notebook never opens an instrument or sends a command.

For a live run, execute this Notebook on the same remote computer as the CLI, click **Connect live run**, then start the CLI in another terminal. A completed or rejected run can be reviewed through **Saved run**.

In [ ]:
import asyncio
from io import BytesIO
import json
from pathlib import Path
from urllib.request import urlopen

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

from attodry_control.three_smu_analysis import discover_three_smu_runs
from attodry_control.three_smu_plot import (
    axis_options, categorical_options, default_plot_specs,
    live_payload_to_plot_sample, load_three_smu_plot_samples,
    plot_map, plot_xy, select_samples,
)

LIVE_ENDPOINT = 'http://127.0.0.1:8765/events'
DEFAULT_DATA_DIRECTORY = Path('../data/three_smu')


In [ ]:
class PlotCard:
    def __init__(self, dashboard, spec=None):
        self.dashboard = dashboard
        spec = spec or {}
        self._updating = False
        self.kind = widgets.Dropdown(
            options=[('Line', 'line'), ('Scatter', 'scatter'), ('2D colour map', 'map')],
            value=spec.get('type', 'line'), description='Type:'
        )
        self.x_axis = widgets.Dropdown(description='X:')
        self.y_axis = widgets.Dropdown(description='Y:')
        self.z_axis = widgets.Dropdown(description='Colour:')
        self.series = widgets.Dropdown(description='Series:')
        self.segment = widgets.Dropdown(description='Segment:')
        self.repeat = widgets.Dropdown(description='Repeat:')
        self.slice_axis = widgets.Dropdown(description='Slice:')
        self.slice_value = widgets.Dropdown(description='Value:')
        self.x_log = widgets.Checkbox(False, description='log X')
        self.y_log = widgets.Checkbox(False, description='log Y')
        self.remove = widgets.Button(description='Remove plot', button_style='danger', icon='trash')
        self.message = widgets.HTML()
        self.image = widgets.Image(format='png', layout=widgets.Layout(max_width='760px'))
        self._requested = spec
        for control in (self.kind, self.x_axis, self.y_axis, self.z_axis, self.series,
                        self.segment, self.repeat, self.slice_axis, self.slice_value,
                        self.x_log, self.y_log):
            control.observe(self._changed, names='value')
        self.remove.on_click(self._remove)
        self.box = widgets.VBox([
            widgets.HTML('<hr><b>Plot</b>'),
            widgets.HBox([self.kind, self.x_axis, self.y_axis, self.z_axis]),
            widgets.HBox([self.series, self.segment, self.repeat]),
            widgets.HBox([self.slice_axis, self.slice_value, self.x_log, self.y_log, self.remove]),
            self.message, self.image,
        ])
        self.refresh()

    def _set_options(self, control, options, requested=None):
        current = requested if requested is not None else control.value
        control.options = options
        values = [value for _label, value in options]
        control.value = current if current in values else (values[0] if values else None)

    def refresh(self):
        self._updating = True
        try:
            axes = axis_options(self.dashboard.samples)
            if not axes:
                axes = [('Elapsed time (s)', 'elapsed_s'), ('Point index', 'point_index')]
            self._set_options(self.x_axis, axes, self._requested.get('x'))
            self._set_options(self.y_axis, axes, self._requested.get('y'))
            self._set_options(self.z_axis, axes, self._requested.get('z'))
            series = [('None', ''), ('Segment', 'segment'), ('Repeat', 'repeat_index'), *axes]
            self._set_options(self.series, series, self._requested.get('series'))
            self._set_options(self.segment, [('All', ''), *categorical_options(self.dashboard.samples, 'segment')])
            self._set_options(self.repeat, [('All', ''), *categorical_options(self.dashboard.samples, 'repeat_index')])
            self._set_options(self.slice_axis, [('None', ''), *axes])
            self._refresh_slice_values()
            self.z_axis.layout.display = '' if self.kind.value == 'map' else 'none'
            self.series.layout.display = 'none' if self.kind.value == 'map' else ''
        finally:
            self._updating = False
        self._requested = {}
        self.draw()

    def _refresh_slice_values(self):
        key = self.slice_axis.value
        values = [] if not key else categorical_options(self.dashboard.samples, key)
        self._set_options(self.slice_value, [('All', ''), *values])
        self.slice_value.layout.display = '' if key else 'none'

    def _changed(self, change):
        if self._updating:
            return
        if change['owner'] is self.slice_axis:
            self._updating = True
            try:
                self._refresh_slice_values()
            finally:
                self._updating = False
        if change['owner'] is self.kind:
            self.z_axis.layout.display = '' if self.kind.value == 'map' else 'none'
            self.series.layout.display = 'none' if self.kind.value == 'map' else ''
        self.draw()

    def _remove(self, _button):
        self.dashboard.remove_plot(self)

    def draw(self):
        if not self.dashboard.samples:
            self.image.value = b''
            self.message.value = 'Waiting for samples. Select a saved run or connect to a live run.'
            return
        try:
            selected = select_samples(
                self.dashboard.samples,
                segment=self.segment.value or None,
                repeat_index=None if not self.repeat.value else int(self.repeat.value),
                slice_axis=self.slice_axis.value or None,
                slice_value=None if not self.slice_value.value else float(self.slice_value.value),
            )
            if self.kind.value == 'map':
                figure = plot_map(selected, x_axis=self.x_axis.value, y_axis=self.y_axis.value,
                                  color_axis=self.z_axis.value, x_log=self.x_log.value, y_log=self.y_log.value)
            else:
                figure = plot_xy(selected, x_axis=self.x_axis.value, y_axis=self.y_axis.value,
                                 series_axis=self.series.value or None, scatter=self.kind.value == 'scatter',
                                 x_log=self.x_log.value, y_log=self.y_log.value)
            png = BytesIO()
            figure.savefig(png, format='png', dpi=130)
            plt.close(figure)
            self.image.value = png.getvalue()
            self.message.value = ''
        except ValueError as exc:
            self.image.value = b''
            self.message.value = f'No plot for this selection: {exc}'


class ThreeSmuDashboard:
    def __init__(self):
        self.samples = ()
        self.mode = None
        self._seen = set()
        self._live_task = None
        self.source = widgets.ToggleButtons(options=[('Saved run', 'saved'), ('Live run', 'live')], description='Source:')
        self.data_directory = widgets.Text(value=str(DEFAULT_DATA_DIRECTORY), description='Data folder:', layout=widgets.Layout(width='520px'))
        self.refresh_runs = widgets.Button(description='Refresh runs', icon='refresh')
        self.run_selector = widgets.Dropdown(description='Run:', layout=widgets.Layout(width='650px'))
        self.audit_rejected = widgets.Checkbox(False, description='Audit: include rejected runs')
        self.audit_problem = widgets.Checkbox(False, description='Audit: include problem samples')
        self.load_saved = widgets.Button(description='Load selected run', button_style='primary', icon='folder-open')
        self.connect_live = widgets.Button(description='Connect live run', button_style='success', icon='play')
        self.stop_live = widgets.Button(description='Stop live view', icon='stop')
        self.status = widgets.HTML()
        self.add_plot_button = widgets.Button(description='Add plot', icon='plus')
        self.auto_layout_button = widgets.Button(description='Apply automatic layout', icon='magic')
        self.cards = []
        self.cards_box = widgets.VBox()
        self.saved_controls = widgets.VBox([
            widgets.HBox([self.data_directory, self.refresh_runs]), self.run_selector,
            widgets.HBox([self.audit_rejected, self.audit_problem, self.load_saved]),
        ])
        self.live_controls = widgets.HBox([self.connect_live, self.stop_live, widgets.HTML(f'<code>{LIVE_ENDPOINT}</code>')])
        self.root = widgets.VBox([
            self.source, self.saved_controls, self.live_controls, self.status,
            widgets.HBox([self.add_plot_button, self.auto_layout_button]), self.cards_box,
        ])
        self.source.observe(self._source_changed, names='value')
        self.refresh_runs.on_click(self._refresh_runs)
        self.load_saved.on_click(self._load_saved)
        self.connect_live.on_click(self._connect_live)
        self.stop_live.on_click(self._stop_live)
        self.add_plot_button.on_click(lambda _button: self.add_plot())
        self.auto_layout_button.on_click(lambda _button: self.apply_auto_layout())
        self._source_changed(None)
        self._refresh_runs(None)

    def show(self):
        display(self.root)

    def _source_changed(self, _change):
        saved = self.source.value == 'saved'
        self.saved_controls.layout.display = '' if saved else 'none'
        self.live_controls.layout.display = 'none' if saved else ''

    def _refresh_runs(self, _button):
        try:
            runs = discover_three_smu_runs(self.data_directory.value, include_rejected=self.audit_rejected.value)
            self.run_selector.options = [(f'{item.started_at} | {item.status} | {item.run_name} | {item.run_dir.name}', str(item.run_dir)) for item in runs]
            self.status.value = f'<b>Saved runs:</b> {len(runs)} found'
        except ValueError as exc:
            self.run_selector.options = []
            self.status.value = f'<b>Saved runs:</b> {exc}'

    def _load_saved(self, _button):
        if not self.run_selector.value:
            self.status.value = '<b>Saved run:</b> select a run first'
            return
        self._stop_live(None)
        self.samples = load_three_smu_plot_samples(
            self.run_selector.value, include_rejected=self.audit_rejected.value,
            include_problem=self.audit_problem.value,
        )
        self.mode = None
        self._seen = {(item.point_index, item.repeat_index, item.segment) for item in self.samples}
        self.status.value = f'<b>Saved run loaded:</b> {len(self.samples)} formal samples (accepted-only unless Audit is checked)'
        self.refresh_cards()
        if not self.cards:
            self.apply_auto_layout()

    def _connect_live(self, _button):
        self._stop_live(None)
        self.samples, self._seen, self.mode = (), set(), None
        self.status.value = '<b>Live run:</b> waiting for the CLI endpoint; no hardware is opened by this Notebook'
        self._live_task = asyncio.create_task(self._follow_live())

    def _stop_live(self, _button):
        if self._live_task is not None:
            self._live_task.cancel()
            self._live_task = None

    async def _follow_live(self):
        while True:
            response = None
            try:
                response = await asyncio.to_thread(urlopen, LIVE_ENDPOINT, timeout=10)
                event_name = None
                while True:
                    raw_line = await asyncio.to_thread(response.readline)
                    if not raw_line:
                        break
                    line = raw_line.decode('utf-8').strip()
                    if line.startswith('event: '):
                        event_name = line[7:]
                    elif line.startswith('data: ') and event_name:
                        if self._live_event(event_name, json.loads(line[6:])):
                            return
            except asyncio.CancelledError:
                return
            except Exception as exc:
                self.status.value = f'<b>Live run:</b> waiting for CLI ({type(exc).__name__}: {exc})'
                await asyncio.sleep(1)
            finally:
                if response is not None:
                    response.close()

    def _live_event(self, name, payload):
        if name == 'run_started':
            self.mode = payload.get('mode')
            self.status.value = f"<b>Live run:</b> {self.mode}; {payload.get('total_samples')} formal samples planned"
            return False
        if name == 'sample':
            item = live_payload_to_plot_sample(payload)
            key = (item.point_index, item.repeat_index, item.segment)
            if key not in self._seen:
                self._seen.add(key)
                self.samples = (*self.samples, item)
                self.status.value = f'<b>Live run:</b> {len(self.samples)} recorded formal samples received'
                self.refresh_cards()
                if not self.cards:
                    self.apply_auto_layout()
            return False
        if name == 'run_finished':
            self.status.value = '<b>Live run:</b> completed; switch to Saved run for accepted-only reload'
            return True
        if name == 'run_failed':
            self.status.value = f"<b>Live run:</b> {payload.get('status')}; retained data is provisional/audit-only ({payload.get('error')})"
            return True
        return False

    def add_plot(self, spec=None):
        card = PlotCard(self, spec)
        self.cards.append(card)
        self.cards_box.children = tuple(item.box for item in self.cards)

    def remove_plot(self, card):
        self.cards.remove(card)
        self.cards_box.children = tuple(item.box for item in self.cards)

    def refresh_cards(self):
        for card in self.cards:
            card.refresh()

    def apply_auto_layout(self):
        self.cards = []
        for spec in default_plot_specs(self.samples, self.mode):
            self.add_plot(spec)


dashboard = ThreeSmuDashboard()
dashboard.show()


## Plotting a family of bias I–V curves at different gate voltages

Add a **Line** plot and choose:

- X: `smu_bias: requested coordinate`
- Y: `smu_bias: current I (A)`
- Series: `gate_top: requested coordinate`
- Slice: `gate_bottom: requested coordinate`, then select one value when the bottom gate also sweeps.

Each gate value becomes one coloured I–V curve and is labelled in the legend. If forward and reverse segments are both present, they remain separate curves by default. `Repeat` filters repeated readings at the same coordinate; it is not another full scan.